In [1]:
import os
import sys 
import time 
import logging
import numpy as np
import pandas as pd 
import geopandas as gpd
from shapely.wkt import loads
from residual_demand_assignment import assignment

In [2]:
abs_path = os.path.dirname(os.path.abspath(__name__))

In [3]:
def main(hour_list=None, quarter_list=None, closure_hours=None):


    ############# CHANGE HERE ############# 
    ### scenario name and input files
    scen_nm = 'Tunnel'

    ### input files
    network_edges = abs_path + '/../projects/test/network_inputs/updated_edges.csv'
    network_nodes = abs_path + '/../projects/test/network_inputs/updated_nodes.csv'
    closed_edges_file = abs_path + '/../projects/test/network_inputs/closed_edges.csv'
    demand_file = abs_path + '/../projects/test/demand_inputs/od.csv'
    simulation_outputs = abs_path + '/../projects/test/simulation_outputs'
 
    ############# NO CHANGE HERE ############# 
    ### network processing
    edges_df = pd.read_csv(network_edges)
    edges_df = edges_df[["uniqueid", "geometry", "osmid", "length", "type", "lanes", "maxspeed", "fft", "capacity", "start_nid", "end_nid"]]
    edges_df = gpd.GeoDataFrame(edges_df, crs='epsg:4326', geometry=edges_df['geometry'].map(loads))
    edges_df = edges_df.sort_values(by='fft', ascending=False).drop_duplicates(subset=['start_nid', 'end_nid'], keep='first')
    ### pay attention to the unit conversion
    edges_df['edge_str'] = edges_df['start_nid'].astype('str') + '-' + edges_df['end_nid'].astype('str')
    edges_df['normal_capacity'] = edges_df['capacity']
    edges_df['normal_fft'] = edges_df['fft']
    edges_df['t_avg'] = edges_df['fft']
    edges_df['u'] = edges_df['start_nid']
    edges_df['v'] = edges_df['end_nid']
    edges_df = edges_df.set_index('edge_str')
    ### closure locations
    closed_links = pd.read_csv(closed_edges_file)
    for row in closed_links.itertuples():
        edges_df.loc[(edges_df['uniqueid']==getattr(row, 'uniqueid')), 'capacity'] = 1
        edges_df.loc[(edges_df['uniqueid']==getattr(row, 'uniqueid')), 'fft'] = 36000
    ### output closed file for visualization
    edges_df.loc[edges_df['fft'] == 36000, ['uniqueid', 'start_nid', 'end_nid', 'capacity', 'fft', 'geometry']].to_csv(simulation_outputs + '/closed_links_{}.csv'.format(scen_nm))

    ### nodes processing
    nodes_df = pd.read_csv(network_nodes)

    nodes_df['x'] = nodes_df['lon']
    nodes_df['y'] = nodes_df['lat']
    nodes_df = nodes_df.set_index('node_id')

    ### demand processing
    t_od_0 = time.time()
    od_all = pd.read_csv(demand_file)
    t_od_1 = time.time()
    logging.info('{} sec to read {} OD pairs'.format(t_od_1-t_od_0, od_all.shape[0]))
    
    ### run residual_demand_assignment
    assignment(edges_df=edges_df, nodes_df=nodes_df, od_all=od_all, simulation_outputs=simulation_outputs, scen_nm=scen_nm, hour_list=hour_list, quarter_list=quarter_list, closure_hours=closure_hours, closed_links=closed_links)

    return True

In [4]:
if __name__ == "__main__":
    
    status = main(hour_list=list(range(7, 9)), quarter_list=[0,1,2,3,4,5], closure_hours=[])

7 0 0
network
net
Generating contraction hierarchies with 10 threads.
Setting CH node vector of size 97
Setting CH edge vector of size 99
Range graph removed 0 edges of 198
. 10% . 20% . 30% . 40% . 50% . 60% . 70% . 80% . 90% . 100%
 100% 7 1 0
network
net
Generating contraction hierarchies with 10 threads.
Setting CH node vector of size 97
Setting CH edge vector of size 99
Range graph removed 0 edges of 198
. 10% . 20% . 30% . 40% . 50% . 60% . 70% . 80% . 90% . 100%
 100% 7 2 0
network
Generating contraction hierarchies with net
10 threads.
Setting CH node vector of size 97
Setting CH edge vector of size 99
Range graph removed 0 edges of 198
. 10% . 20% . 30% . 40% . 50% . 60% . 70% . 80% . 90% . 100%
 100% 7 3 0
Generating contraction hierarchies with 10 threads.
Setting CH node vector of size 97
Setting CH edge vector of size 99
network
net
Range graph removed 0 edges of 198
. 10% . 20% . 30% . 40% . 50% . 60% . 70% . 80% . 90% . 100%
 100% 7 4 0
Generating contraction hierarchies

### Travel Time

In [5]:
df = pd.read_csv('../projects/test/simulation_outputs/trip_info/trip_info_Tunnel.csv')

In [6]:
df

,agent_id,origin_nid,destin_nid,travel_time,travel_time_used,stop_nid,stop_hour,stop_quarter,stop_ssid
0,0,1018,46,1200.0,1011.710000,46,7,2,0
1,1,1018,46,1200.0,1011.710000,46,7,2,0
2,2,1018,46,600.0,319.757256,46,7,0,0
3,3,1018,46,1200.0,991.650000,46,8,0,0
4,4,1018,46,1200.0,835.540000,46,7,5,0
...,...,...,...,...,...,...,...,...,...
14996,14996,1018,46,1200.0,991.650000,46,8,0,0
14997,14997,1018,46,600.0,319.757256,46,7,0,0
14998,14998,1018,46,600.0,319.757256,46,7,0,0
14999,14999,1018,46,1200.0,991.650000,46,8,0,0


In [7]:
df['travel_time_used'].mean()

838.1321712973337

In [8]:
df['travel_time_used'].unique()

array([1011.71      ,  319.75725563,  991.65      ,  835.54      ,
       1180.17      ,  686.44      ])